In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from os.path import join
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.models as models
from MaskRefiner import MaskRefinerDecoder
from tqdm import tqdm
from torch.utils.data import random_split, DataLoader, Dataset
from dataset import MyDataset
import segmentation_models_pytorch as smp
from os.path import split
from pycocotools import mask as mask_utils
from json import load
from torch import nn
import gc

# Enable cudNN benchmarking for faster training
torch.backends.cudnn.benchmark = True

In [ ]:
# GPU Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    # Enable mixed precision training for faster computation
    use_amp = True
    scaler = torch.cuda.amp.GradScaler()
else:
    use_amp = False
    print("Warning: GPU not available, using CPU")

In [ ]:
# Paths - Update these for Colab if needed
input_dir = "/Users/carricarte/PhD/Projects/MARL-med/Repo/Multiagent_RL/scratch/dataset"
out_dir = "/Users/carricarte/PhD/Projects/MARL-med/scratch/output"
csv_path = "/Users/carricarte/PhD/Projects/MARL-med/Repo/Multiagent_RL/scratch/dataset"

# Create output directory if it doesn't exist
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# Model Configuration
in_channels = 768
base_channels = 64
correction_scale = 0.3
dropout_rate = 0.1
use_fusion = True
use_multi_scale = False

my_model = MaskRefinerDecoder(
    embed_channels=in_channels,
    base_channels=base_channels,
    correction_scale=correction_scale,
    use_mask_fusion=use_fusion,
    dropout_rate=dropout_rate
).to(device)  # Move model to GPU

print(f"Model parameters: {sum(p.numel() for p in my_model.parameters()):,}")

In [ ]:
def decode_mask_format(rle_dict, device):
    """
    Decode RLE and prepare for MaskRefiner input.
    
    Args:
        rle_dict: RLE dictionary with 'counts' and 'size'
        device: torch device
    
    Returns:
        Torch tensor [1, 1, H, W] ready for model
    """
    binary_mask = mask_utils.decode(rle_dict)  # [H, W]
    mask_tensor = torch.from_numpy(binary_mask).float()
    mask_tensor = mask_tensor.unsqueeze(0).unsqueeze(0)
    mask_tensor = mask_tensor.to(device)
    
    return mask_tensor


def plot_training_history(train_losses, val_losses):
    """Plot training history"""
    fig, ax1 = plt.subplots(1, figsize=(10, 6))
    
    ax1.plot(train_losses, label='Train Loss', linewidth=2)
    ax1.plot(val_losses, label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_path = join(out_dir, 'training_history.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Training history saved to {save_path}")
    plt.show()


def clear_gpu_memory():
    """Clear GPU cache to prevent OOM errors"""
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# Dataset preparation
print("Loading dataset...")
my_dataset = MyDataset(input_dir)
print(f"Total samples: {len(my_dataset)}")

# Split dataset
train_size = int(0.7 * len(my_dataset))
val_size = int(0.15 * len(my_dataset))
test_size = len(my_dataset) - train_size - val_size  # Ensure exact split

my_train_dataset, my_val_dataset, my_test_dataset = random_split(
    my_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)  # Reproducibility
)

print(f"Train: {train_size}, Val: {val_size}, Test: {test_size}")

In [ ]:
# Optimized DataLoader settings
batch_size = 32 if torch.cuda.is_available() else 8  # Adjust based on GPU memory
num_workers = 2 if torch.cuda.is_available() else 0  # More workers can cause issues in Colab

my_train_loader = DataLoader(
    my_train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False,  # Faster GPU transfer
    persistent_workers=True if num_workers > 0 else False
)

my_val_loader = DataLoader(
    my_val_dataset, 
    batch_size=batch_size, 
    shuffle=False,  # No need to shuffle validation
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if num_workers > 0 else False
)

my_test_loader = DataLoader(
    my_test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if num_workers > 0 else False
)

print(f"Batch size: {batch_size}")
print(f"Train batches: {len(my_train_loader)}")
print(f"Val batches: {len(my_val_loader)}")

In [ ]:
# Training configuration
criterion = smp.losses.FocalLoss(mode='binary', alpha=0.25, gamma=2.0)
optimizer = torch.optim.AdamW(
    my_model.parameters(), 
    lr=5e-4,
    weight_decay=1e-4  # L2 regularization
)

# Learning rate scheduler for better convergence
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=3,
    verbose=True
)

epochs = 15
best_val_loss = float('inf')
patience_counter = 0
early_stop_patience = 7

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, use_amp, scaler=None):
    """Optimized training epoch with GPU support and mixed precision"""
    model.train()
    losses = []
    
    pbar = tqdm(train_loader, desc='Training')
    for embedding, p_segmentation, gt_segmentation in pbar:
        # Move data to GPU
        embedding = embedding.to(device, non_blocking=True)
        p_segmentation = p_segmentation.to(device, non_blocking=True)
        gt_segmentation = gt_segmentation.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()
        
        # Mixed precision training
        if use_amp:
            with torch.cuda.amp.autocast():
                refined_mask = model(embedding, p_segmentation)
                loss = criterion(
                    refined_mask.contiguous().view(-1), 
                    gt_segmentation.contiguous().view(-1)
                )
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            refined_mask = model(embedding, p_segmentation)
            loss = criterion(
                refined_mask.contiguous().view(-1), 
                gt_segmentation.contiguous().view(-1)
            )
            loss.backward()
            optimizer.step()
        
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return np.mean(losses)


def val_epoch(model, val_loader, criterion, device, use_amp):
    """Optimized validation epoch"""
    model.eval()
    losses = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for embedding, p_segmentation, gt_segmentation in pbar:
            # Move data to GPU
            embedding = embedding.to(device, non_blocking=True)
            p_segmentation = p_segmentation.to(device, non_blocking=True)
            gt_segmentation = gt_segmentation.to(device, non_blocking=True)
            
            if use_amp:
                with torch.cuda.amp.autocast():
                    refined_mask = model(embedding, p_segmentation)
                    loss = criterion(
                        refined_mask.contiguous().view(-1), 
                        gt_segmentation.contiguous().view(-1)
                    )
            else:
                refined_mask = model(embedding, p_segmentation)
                loss = criterion(
                    refined_mask.contiguous().view(-1), 
                    gt_segmentation.contiguous().view(-1)
                )
            
            losses.append(loss.item())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return np.mean(losses)

In [ ]:
# Training loop with early stopping and checkpointing
train_losses = []
val_losses = []

print("\nStarting training...\n")

for epoch in range(epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"{'='*60}")
    
    # Training
    train_loss = train_epoch(
        my_model, my_train_loader, criterion, optimizer, 
        device, use_amp, scaler if use_amp else None
    )
    train_losses.append(train_loss)
    
    # Validation
    val_loss = val_epoch(my_model, my_val_loader, criterion, device, use_amp)
    val_losses.append(val_loss)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Print summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        checkpoint_path = join(out_dir, 'best_model.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': my_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, checkpoint_path)
        print(f"✓ New best model saved! (Val Loss: {val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter}/{early_stop_patience}")
    
    # Early stopping
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping triggered after {epoch + 1} epochs")
        break
    
    # Clear GPU cache periodically
    if (epoch + 1) % 5 == 0:
        clear_gpu_memory()

print("\n" + "="*60)
print("Training completed!")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*60)

# Plot training history
plot_training_history(train_losses, val_losses)

In [ ]:
# Load best model for evaluation
checkpoint = torch.load(join(out_dir, 'best_model.pth'))
my_model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")

# Test evaluation
test_loss = val_epoch(my_model, my_test_loader, criterion, device, use_amp)
print(f"\nTest Loss: {test_loss:.4f}")